# Pandas 全面学习

Pandas 是 Python 数据处理的核心库，提供 DataFrame（表格）和 Series（列）两种数据结构，是数据清洗、分析、特征工程的必备工具。

In [ ]:
import pandas as pd
import numpy as np
print(f"Pandas 版本: {pd.__version__}")

---
## 1. Series 与 DataFrame 基础

### 1.1 Series：带标签的一维数组

In [ ]:
# 创建 Series
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'])
print(s)
print(f"\n值: {s.values}")
print(f"索引: {s.index.tolist()}")

# 按索引取值
print(f"\ns['b'] = {s['b']}")
print(f"s[['a','c']] = \n{s[['a', 'c']]}")

### 1.2 DataFrame：带行列标签的二维表格

In [ ]:
# 从字典创建 DataFrame
data = {
    '姓名': ['张三', '李四', '王五', '赵六'],
    '年龄': [20, 22, 19, 21],
    '成绩': [85, 92, 78, 88],
    '城市': ['北京', '上海', '广州', '深圳']
}
df = pd.DataFrame(data)
print(df)
print(f"\n形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")
print(f"索引: {df.index.tolist()}")

---
## 2. 数据查看与基本信息

In [ ]:
# 创建稍大的示例数据
np.random.seed(42)
n = 100
df_demo = pd.DataFrame({
    '学号': range(1, n+1),
    '姓名': [f'学生{i}' for i in range(1, n+1)],
    '数学': np.random.normal(75, 12, n).clip(0, 100).astype(int),
    '英语': np.random.normal(70, 15, n).clip(0, 100).astype(int),
    '物理': np.random.normal(68, 18, n).clip(0, 100).astype(int),
    '性别': np.random.choice(['男', '女'], n),
    '班级': np.random.choice(['A班', 'B班', 'C班'], n)
})

# 查看前几行
print("=== head() ===")
print(df_demo.head())

# 查看后几行
print("\n=== tail(3) ===")
print(df_demo.tail(3))

In [ ]:
# 数据类型和缺失值
print("=== info() ===")
df_demo.info()

# 数值列统计
print("\n=== describe() ===")
print(df_demo.describe())

---
## 3. 数据选择

### 3.1 选择列

In [ ]:
# 选单列（返回 Series）
print(type(df_demo['姓名']))
print(df_demo['姓名'].head())

# 选多列（返回 DataFrame）
print("\n选多列:")
print(df_demo[['姓名', '数学', '英语']].head())

### 3.2 loc vs iloc（重点）

In [ ]:
# loc：按标签索引
print("loc[0:3]:")
print(df_demo.loc[0:3, ['姓名', '数学']])  # 注意：包含右端

# iloc：按位置索引（从0开始）
print("\niloc[0:3]:")
print(df_demo.iloc[0:3, [1, 2]])  # 不包含右端

# iloc 取行+列范围
print("\niloc[0:5, 2:5]:")
print(df_demo.iloc[0:5, 2:5])

### 3.3 条件筛选

In [ ]:
# 单条件
high_math = df_demo[df_demo['数学'] >= 90]
print(f"数学>=90: {len(high_math)}人")

# 多条件（& 且, | 或）
good_students = df_demo[(df_demo['数学'] >= 85) & (df_demo['英语'] >= 85)]
print(f"数学英语都>=85: {len(good_students)}人")
print(good_students.head())

# isin 筛选
class_ab = df_demo[df_demo['班级'].isin(['A班', 'B班'])]
print(f"\nAB班人数: {len(class_ab)}")

---
## 4. 数据清洗

In [ ]:
# 创建含有缺失值的数据
df_dirty = pd.DataFrame({
    '姓名': ['张三', '李四', '王五', '赵六', '钱七'],
    '年龄': [20, np.nan, 19, 21, 22],
    '成绩': [85, 92, np.nan, 88, 76],
    '城市': ['北京', '上海', None, '深圳', '上海']
})
print("原始数据:")
print(df_dirty)

# 检查缺失值
print(f"\n各列缺失值:\n{df_dirty.isnull().sum()}")

In [ ]:
# 方式1: 删除含缺失值的行
print("dropna 后:")
print(df_dirty.dropna())

# 方式2: 填充缺失值
df_filled = df_dirty.copy()
df_filled['年龄'].fillna(df_filled['年龄'].mean(), inplace=True)
df_filled['成绩'].fillna(df_filled['成绩'].median(), inplace=True)
df_filled['城市'].fillna('未知', inplace=True)
print("\n填充后:")
print(df_filled)

### 4.1 数据类型转换

In [ ]:
df_type = pd.DataFrame({
    '价格': ['100', '200', '300'],
    '日期': ['2024-01-01', '2024-02-01', '2024-03-01'],
    '是否促销': ['是', '否', '是']
})
print("转换前:")
print(df_type.dtypes)

df_type['价格'] = df_type['价格'].astype(float)
df_type['日期'] = pd.to_datetime(df_type['日期'])
df_type['是否促销'] = df_type['是否促销'].map({'是': True, '否': False})

print("\n转换后:")
print(df_type.dtypes)

---
## 5. 数据操作

### 5.1 添加/修改列

In [ ]:
df_calc = df_demo[['姓名', '数学', '英语', '物理']].head().copy()

# 添加新列
df_calc['总分'] = df_calc['数学'] + df_calc['英语'] + df_calc['物理']
df_calc['均分'] = df_calc[['数学', '英语', '物理']].mean(axis=1).round(1)
df_calc['等级'] = pd.cut(df_calc['均分'], bins=[0, 60, 80, 100], 
                        labels=['不及格', '良好', '优秀'])
print(df_calc)

### 5.2 排序

In [ ]:
# 按成绩降序
print("按数学降序前5:")
print(df_demo.nlargest(5, '数学')[['姓名', '数学', '英语']])

# 多列排序
print("\n按班级升序、数学降序:")
print(df_demo.sort_values(['班级', '数学'], ascending=[True, False]).head(10)[['姓名', '班级', '数学']])

### 5.3 分组聚合（GroupBy）

In [ ]:
# 按班级分组统计
class_stats = df_demo.groupby('班级')[['数学', '英语', '物理']].agg(['mean', 'std', 'max'])
print("班级成绩统计:")
print(class_stats.round(1))

# 按性别分组
gender_stats = df_demo.groupby('性别')[['数学', '英语']].mean()
print("\n性别成绩均值:")
print(gender_stats.round(1))

In [ ]:
# 自定义聚合
result = df_demo.groupby('班级').agg(
    人数=('姓名', 'count'),
    数学均分=('数学', 'mean'),
    数学最高=('数学', 'max'),
    数学及格率=('数学', lambda x: (x >= 60).mean() * 100)
).round(1)
print(result)

---
## 6. 数据合并

In [ ]:
# concat：上下拼接
df1 = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})
df2 = pd.DataFrame({'A': [5, 6], 'B': [7, 8]})
print("concat 上下拼接:")
print(pd.concat([df1, df2], ignore_index=True))

# merge：类似 SQL 的 join
students = pd.DataFrame({
    '学号': [1, 2, 3],
    '姓名': ['张三', '李四', '王五']
})
scores = pd.DataFrame({
    '学号': [1, 2, 4],
    '成绩': [85, 92, 78]
})

print("\ninner merge:")
print(pd.merge(students, scores, on='学号', how='inner'))

print("\nleft merge:")
print(pd.merge(students, scores, on='学号', how='left'))

---
## 7. 读写文件

In [ ]:
import os

# 写入 CSV
df_demo.head(20).to_csv('students_sample.csv', index=False, encoding='utf-8-sig')
print("已保存 students_sample.csv")

# 读取 CSV
df_read = pd.read_csv('students_sample.csv')
print(f"读取成功: {df_read.shape}")
print(df_read.head())

# 清理
os.remove('students_sample.csv')

In [ ]:
# 读取 Excel（需要 openpyxl）
# df = pd.read_excel('data.xlsx', sheet_name='Sheet1')
# df.to_excel('output.xlsx', index=False)

# 读取 JSON
df_demo.head(5).to_json('students_sample.json', orient='records', force_ascii=False)
df_json = pd.read_json('students_sample.json')
print("JSON 读取:")
print(df_json.head())
os.remove('students_sample.json')

---
## 8. 实战案例：电商销售数据分析

In [ ]:
np.random.seed(42)
n = 500

dates = pd.date_range('2024-01-01', periods=365, freq='D')
categories = ['手机', '电脑', '耳机', '平板', '手表']
cities = ['北京', '上海', '广州', '深圳', '杭州']

df_sales = pd.DataFrame({
    '日期': np.random.choice(dates, n),
    '商品': np.random.choice(categories, n),
    '城市': np.random.choice(cities, n),
    '数量': np.random.randint(1, 10, n),
    '单价': np.random.choice([2999, 5999, 899, 3999, 1999], n),
    '评分': np.random.choice([3.0, 3.5, 4.0, 4.5, 5.0], n)
})
df_sales['销售额'] = df_sales['数量'] * df_sales['单价']

print(f"数据量: {len(df_sales)} 条")
print(df_sales.head())

In [ ]:
# 1. 各商品销售额排名
product_rank = df_sales.groupby('商品')['销售额'].sum().sort_values(ascending=False)
print("商品销售额排名:")
print(product_rank)

# 2. 各城市销售情况
city_stats = df_sales.groupby('城市').agg(
    总销售额=('销售额', 'sum'),
    订单数=('销售额', 'count'),
    平均评分=('评分', 'mean')
).sort_values('总销售额', ascending=False)
print("\n城市销售统计:")
print(city_stats.round(1))

# 3. 月度趋势
df_sales['月份'] = df_sales['日期'].dt.to_period('M')
monthly = df_sales.groupby('月份')['销售额'].sum()
print("\n月度销售额:")
print(monthly)

---
## 总结

| 知识点 | 关键方法 |
|--------|----------|
| 创建 | `DataFrame()`, `Series()` |
| 查看 | `head`, `tail`, `info`, `describe`, `shape` |
| 选择 | `[]`, `loc`, `iloc`, 布尔索引 |
| 清洗 | `isnull`, `dropna`, `fillna`, `astype`, `duplicated` |
| 操作 | `assign`, `sort_values`, `value_counts` |
| 聚合 | `groupby`, `agg`, `pivot_table` |
| 合并 | `concat`, `merge`, `join` |
| IO | `read_csv`, `to_csv`, `read_excel`, `to_excel` |

**下一步**：学习 Matplotlib/Seaborn 进行数据可视化 → `Matplotlib_Seaborn全面学习.ipynb`